This code was adapted from: https://github.com/nliulab/mimic4ed-benchmark

In [ ]:
import os
import time
import random
import numpy as np
import pandas as pd

df_train = pd.read_csv(f'data/preprocessed/train.csv')
df_test = pd.read_csv(f'data/preprocessed/test.csv')
df_test_included = pd.read_pickle(f'data/preprocessed/baseline_synthetic_data_3_3std_no0_len3.pkl')

df_test_included['stay_id'] = df_test_included['stay_id'].astype(int)
df_test = df_test[df_test['stay_id'].isin(df_test_included['stay_id'])]

In [ ]:
random_seed=0
random.seed(random_seed)
np.random.seed(random_seed)

In [ ]:
df_train['target_los'] = (df_train['ed_los_hours'] > 4).astype(int)
df_test['target_los'] = (df_test['ed_los_hours'] > 4).astype(int)

In [ ]:
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)
df_train.head()

In [ ]:
print('training size =', len(df_train), ', testing size =', len(df_test))

In [ ]:
variable = ["age",

            "n_ed_30d", "n_ed_90d", "n_ed_365d", "n_hosp_30d", "n_hosp_90d",
            "n_hosp_365d", "n_icu_30d", "n_icu_90d", "n_icu_365d",

            "triage_temperature", "triage_heartrate", "triage_resprate",
            "triage_o2sat", "triage_sbp", "triage_dbp", "triage_pain", "triage_acuity",

            "chiefcom_chest_pain", "chiefcom_abdominal_pain", "chiefcom_headache",
            "chiefcom_shortness_of_breath", "chiefcom_back_pain", "chiefcom_cough",
            "chiefcom_nausea_vomiting", "chiefcom_fever_chills", "chiefcom_syncope",
            "chiefcom_dizziness",

            "cci_MI", "cci_CHF", "cci_PVD", "cci_Stroke", "cci_Dementia",
            "cci_Pulmonary", "cci_Rheumatic", "cci_PUD", "cci_Liver1", "cci_DM1",
            "cci_DM2", "cci_Paralysis", "cci_Renal", "cci_Cancer1", "cci_Liver2",
            "cci_Cancer2", "cci_HIV",

            "eci_Arrhythmia", "eci_Valvular", "eci_PHTN",  "eci_HTN1", "eci_HTN2",
            "eci_NeuroOther", "eci_Hypothyroid", "eci_Lymphoma", "eci_Coagulopathy",
            "eci_Obesity", "eci_WeightLoss", "eci_FluidsLytes", "eci_BloodLoss",
            "eci_Anemia", "eci_Alcohol", "eci_Drugs","eci_Psychoses", "eci_Depression"]

outcome = "target_los"

In [ ]:
X_train = df_train[variable].copy()
y_train = df_train[outcome].copy()
X_test = df_test[variable].copy()
y_test = df_test[outcome].copy()

In [ ]:
print('RandomForest:')
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(random_state=random_seed)
start = time.time()
rf.fit(X_train,y_train)
runtime = time.time()-start
print('Training time:', runtime, 'seconds')
probs = rf.predict_proba(X_test)
importances = rf.feature_importances_
print(importances)

In [ ]:
print('GradientBoosting:')
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(random_state=random_seed)
start = time.time()
gb.fit(X_train, y_train)
runtime = time.time()-start
print('Training time:', runtime, 'seconds')
probs = gb.predict_proba(X_test)

In [ ]:
print('MLP (sklearn):')
from sklearn.neural_network import MLPClassifier
import time

mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation='relu',
    solver='adam',
    learning_rate_init=0.001,
    batch_size=200,
    max_iter=200,
    alpha=0.0,
    early_stopping=False,
    random_state=random_seed
)

start = time.time()
mlp.fit(X_train, y_train)
runtime = time.time() - start
print('Training time:', runtime, 'seconds')

probs = mlp.predict_proba(X_test)[:, 1]

In [ ]:
# Get ML performance for real test set
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix
from sklearn.metrics import precision_score, recall_score, f1_score

for model_id, name in zip([rf, gb, mlp], ['rf', 'gb', 'mlp']):
    print(name)
    pred_label = model_id.predict(X_test)

    y_prob = model_id.predict_proba(X_test)

    precision = precision_score(y_test, pred_label, average='binary')
    recall = recall_score(y_test, pred_label, average='binary')
    f1 = f1_score(y_test, pred_label, average='binary')

    tn, fp, fn, tp = confusion_matrix(y_test, pred_label).ravel()
    missed_per_100 = fn / (tn + fp + fn + tp) * 100

    print(np.round(precision, 2), np.round(recall, 2), np.round(missed_per_100, 2))

In [ ]:
# Get ML performance for synthetic test sets
folder_list = ['real_normal', 'real_mci_arrival_a', 'real_mci_arrival_b', 'real_mci_arrival_c', 'real_mci_arrival_d', 'real_mci_resource_a', 'real_mci_resource_b', 'real_mci_resource_c', 'real_mci_resource_d', 'real_mci_workflow_a', 'real_mci_workflow_b', 'real_mci_workflow_c', 'real_mci_workflow_d']

for real_exp_folder in folder_list:
    print(f'## {real_exp_folder}')

    for model_id, name in zip([rf, gb, mlp], ['rf', 'gb', 'mlp']):
        precision_list = []
        recall_list = []
        false_list = []
        for i in range(1,1001):
            real_output_folder = f'output_{i}'
            df_syn1 = pd.read_csv(f'experiments/{real_exp_folder}/{real_output_folder}/synthetic_ehr.csv', dtype={'acuity': str})

            df_syn1['new_ed_los_hours'] = df_syn1['ed_los'] / 60
            df_syn1 = df_syn1.rename(columns={"mimic_id": "stay_id"})

            merged_df = df_test.merge(df_syn1, on='stay_id', how='left')
            df_sub1 = merged_df[merged_df['stay_id'].isin(df_syn1['stay_id'])].copy()
            df_sub1['target_los'] = (df_sub1['new_ed_los_hours'] > 4).astype(int)

            X_sub1 = df_sub1[variable].copy()
            y_sub1 = df_sub1[outcome].copy()

            # Predict patient outcomes
            pred_label = model_id.predict(X_sub1)

            precision = precision_score(y_sub1, pred_label, average='binary')
            recall = recall_score(y_sub1, pred_label, average='binary')
            precision_list.append(precision)
            recall_list.append(recall)

            tn, fp, fn, tp = confusion_matrix(y_sub1, pred_label).ravel()
            missed_per_100 = fn / (tn + fp + fn + tp) * 100
            false_list.append(missed_per_100)

        print(name)
        print('precision', np.round(np.mean(precision_list), 2), np.round(np.std(precision_list)*2, 2))
        print('recall', np.round(np.mean(recall_list), 2), np.round(np.std(recall_list)*2, 2))
        print('false', np.round(np.mean(false_list), 2), np.round(np.std(false_list)*2, 2))